### My Prototyping
- Get the continuous text
- Process with datacollator 
- Train!
- Quick review - what are the parameters for tokenizers formed by AutoTokenizer in Hugging Face
    - We can customize tokenization with various parameters ...
    - Common Parameters:
        - `text`  The text to tokenize.
        - `truncation=True`  Cuts off text at `max_length` (default is `False`).
        - `max_length=1024`  Maximum sequence length (used when `truncation=True`).
        - `padding="longest"`  Pads to the longest sequence in the batch.
        - `return_tensors="pt"`  Returns PyTorch (`"pt"`), TensorFlow (`"tf"`), or NumPy (`"np"`) tensors.
        - `return_token_type_ids=False`  Whether to return token type IDs (**needed for BERT, not for Mistral/GPT-2**).
        - `return_attention_mask=True`  Adds an attention mask (1s for real tokens, 0s for padding).
        - `add_special_tokens=True`  Includes special tokens (`<s>`, `</s>` for Mistral, `[CLS]`, `[SEP]` for BERT).
        - `return_length=True`  Returns the number of tokens in the output. 
    - Advanced Parameters
        - `padding="max_length"`  Pads all sequences to the `max_length` value.
        - `stride=128`  Enables sliding window tokenization by keeping an overlap of 128 tokens. Used when splitting long texts into overlapping chunks.
        - `return_offsets_mapping=True`  Returns character-level mappings of tokens.
        - `return_overflowing_tokens=True`  Splits long text into multiple sequences when `max_length` is exceeded.
        - `return_special_tokens_mask=True`  Returns a mask indicating special tokens.
        - `is_split_into_words=True`  Indicates that the input text is **already split into words** (for tokenizing pre-tokenized text).
        - `verbose=False`  Suppresses tokenization warnings.
- Questions:
    - **Where should I split into Training/Testing?**

### Data Cleaning 

In [ ]:
import codecs
# Load in cleaned book text file
blood_memory_txt = "/projectnb/scottml/seansal2/data/clean/blood_memory_clean.txt"
with codecs.open(blood_memory_txt, 'r', encoding='utf-8-sig') as file:
    clean_text = file.read()

In [ ]:
clean_text[0:100]

In [ ]:
clean_text[-100:]

In [ ]:
type(clean_text)

In [ ]:
# Create a HuggingFace dataset
from datasets import Dataset 
dataset = Dataset.from_dict({"text": [clean_text]})
print(dataset)

In [ ]:
# Save dataset for future use
dataset.save_to_disk("/projectnb/scottml/seansal2/data/datasets/blood_memory_clm")

### Model URLs:
- Meta Llama Models
    - Make sure you have access token
    - Llama 2: https://download.llamameta.net/*?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiOXd2NzBxdDl5MHZ6aGltNWVqemtrcmJtIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZG93bmxvYWQubGxhbWFtZXRhLm5ldFwvKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0Mjc3MjQ2OH19fV19&Signature=GECjGtuWGl4uN6fXwGdGIaf5JLFgTdh4gA8Srh50GJ5paaGlGCRaRylsyFvNlCUNfhwnwAvnyxA1waHrLSnFw6ndClXy21P-Hzu8Wb8nLeCJSRgnvGMyQKN8yg0YUQAqrB8m1RgdSpsBz0P4tnZu0EjHTubUqw-vRV4Ey6nmSKuEEvqdBYr3%7EfDu9bmHsjdJ91sGVz4SRQpWHApTSj7%7EK9Pt%7EYcpWgv8OPlUwG-FOi67I5GhKW1a607LzhnHDKuroq-MJbUd1i4EKlHRCOhkO2ogIBSEpYLIpt5R5FHRcVIvtMUY3nk5X9Khs4YtgMI6XqInplifgwgQFpLowX6crQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=671000878807835
    - Llama 3.1 (405B and 8B): https://llama3-1.llamameta.net/*?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoieXloZ240N2Q3bXdiOHgyYjZiamtyajdxIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvbGxhbWEzLTEubGxhbWFtZXRhLm5ldFwvKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0Mjc3MjQ2OH19fV19&Signature=A4CkItFPmrDi76uyaFpEPphLAerbfAGla3wlAx37q08iItEbkmiapK2YY0Fad5xCb62ZDxxWZTMQ68ZfjzAxYPc2r9pg5cC56YdShRz7pVy6QHSThI0LkHFTNDly0L7e42wycAquuF4aiF9f3Yr9vvKmecFmQs6eTSNHQS3fmmlFR29RdCppruDRhsegc9tqzVghfiKzD8NLfVHceLoyh2z9jtr6nUBEm%7E9gRAgelCKnJPtGsUiNb5bnnAza-pn60Lnw903xMM9OfpG9J4RW33PIO-rOJbXpYyZis7eMS%7ETTyuasttdcVdkeM7Tcagk4o4HyXGXg7-ZRjAPnka4xVg__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=2101260510323245
- Future notes
    - Turn this notebook into a script
    - Set up a config file
    - Store tokens + needed credentials in an environment
    - Define special cases for models that require extra credentials

### Config/Setup

In [1]:
#Chceck where models are downloaded
import os
os.environ["HF_HOME"] = "/projectnb/scottml/seansal2/marthabot/model_downloads"

from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
import torch
print(torch.version.cuda)

11.8


In [5]:
# Handle Torch Dynamo Error
import torch._dynamo
torch._dynamo.config.suppress_errors = True


In [6]:
# HuggingFace/LLama login - use my read access token
from huggingface_hub import notebook_login
notebook_login()

In [7]:
# Load in Data
from datasets import load_from_disk
dataset = load_from_disk("/projectnb/scottml/seansal2/data/datasets/blood_memory_clm")
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 1
})


### Tokenization, Pre-Processing

In [8]:
# Follow CLM articles. Do a trial run with 3 different models

# GPT2 - starter model
model_gpt2 = "gpt2"

# Llama Model 
model_llama2_7b = "meta-llama/Llama-2-7b-hf" 

# Mistral Model 
model_mistral_7b = "mistralai/Mistral-7B-v0.1"

# Pick a model 
model_name = model_llama2_7b 

import os
tmp_dir = os.environ.get("TMPDIR")
cache_dir = os.path.join(tmp_dir, "seansal")
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
    print(f"Create {cache_dir}")
else:
    print(f"Directory {cache_dir} exists")
    
# Set where models are downloaded and stored --> avoid redownloading 
# cache_dir = "/projectnb/scottml/seansal2/marthabot/model_downloads"


#Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=torch.float16,  
    device_map="auto"
)

2025-03-26 16:53:05.775820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743022386.171751  458462 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743022386.219263  458462 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743022387.575336  458462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743022387.575383  458462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743022387.575385  458462 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
# Tokenize text!
# First set padding token - tells tokenizer to use end of sentence token to fill in extra space where needed
tokenizer.pad_token = tokenizer.eos_token

# Define tokenization function
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=False, padding=False, return_token_type_ids=False)

# Apply tokenization
tokenized_dataset = dataset.map(preprocess_function, batched=True, num_proc=1)

# Print a sample tokenized output
print(tokenized_dataset)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1
})


In [9]:
# Let's check how many tokens - so we can get a general idea of how many chunks we will form
num_tokens = len(tokenized_dataset[0]["input_ids"])
print(f"Total tokens in dataset: {num_tokens}")

Total tokens in dataset: 94376


##### Block size/chunk length/context_window considerations
- How many tokens are we going to feed the model at once?
- Things to consider...
    - Maximum context window/length of the model
    - Training Efficiency
        - Smaller block sizes (512-1024) --> more samples per batch --> improved training speed and stability
            - Good for short-form data like tweets or individual sentences.
        - Larger block sizes (2048-4096+) --> better for long-form coherence (books, convos, articles)
            - Slower, more memory-intensive
    - Memory (GPU) Constraints
        - Larger Block sizes = higher memory usage
        - Depends what RAM is in our GPU
    - Dataset Nature
        - Short texts - tweets/single sentences --> smaller block sizes to avoid just padding everything
        - Long documents - articles, books --> larger blocks to help model better long-range dependencies and relationships
    - Tokenizer Overhead --> tokens != words.
    - Padding and Efficiency
        - If block size is too big and data is too short --> waste compute on padding
- Block size vs batch size
    - Block size = how many models go into model at once per sample. Length of text chunk we're feeeding in.
    - Batch size = how many of those samples/blocks are processed together in parallel.  

In [10]:
# Group Tokenized text into block_size chunks depeneding on context window and resources (GPU RAM)
# GPT2 - max is 1024, Llama 2 7B- max is 4096, Mistral 7B - max is 8192
block_size = 1024 

def group_texts(examples):
    """
    Groups tokenized text into fixed-size chunks of `block_size`.

    - First, it concatenates all tokenized sequences into a single list.
    - Then, it ensures only full `block_size` chunks are created.
    - Finally, it splits the text into equal-length chunks for model training.
    """
    # Ensure tokenized output is a list of lists 
    # - If examples[k] is already a list of lists (multiple sequences), sum() flattens it
    # - If it's just a single sequence (list of tokens), keep it as is
    concatenated_examples = {k: sum(examples[k], []) if isinstance(examples[k][0], list) else examples[k] for k in examples.keys()}

    # Get total number of tokens after concatenation
    total_length = len(concatenated_examples["input_ids"])

    #Avoid incomplete chunks
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size  # Trim to block_size

    # Split into fixed-size chunks
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    #Set labels = input_ids for CLM
    result["labels"] = result["input_ids"].copy()
    return result

# Apply grouping to create chunked dataset
chunked_dataset = tokenized_dataset.map(group_texts, batched=True, num_proc=1)


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [11]:
# Make sure we didn't lose a lot of tokens
print(f"Number of tokens in dataset: {sum(len(item['input_ids']) for item in chunked_dataset)}")

Number of tokens in dataset: 94208


In [12]:
# Check if grouping worked
# Check the lengths of first few sequences
for i in range(5):  # Check first 5 examples
    print(f"Example {i+1}: Length = {len(chunked_dataset[i]['input_ids'])}")

Example 1: Length = 1024
Example 2: Length = 1024
Example 3: Length = 1024
Example 4: Length = 1024
Example 5: Length = 1024


In [13]:
# Each chunk should be a separate row now:
print(f"Total number of chunks: {len(chunked_dataset)}")

#Check last chunk's length:
print(f"Last chunk length: {len(chunked_dataset[-1]['input_ids'])}")

Total number of chunks: 92
Last chunk length: 1024


In [14]:
# See how many tokens were lost.
total_tokens_after_grouping = sum(len(chunk["input_ids"]) for chunk in chunked_dataset)
print(f"Total tokens after grouping: {total_tokens_after_grouping}")
print(f"{num_tokens - total_tokens_after_grouping} tokens were lost during grouping.")

Total tokens after grouping: 94208
168 tokens were lost during grouping.


In [15]:
# Now that we have chunks, split data into training and testing, start with 90/10 split
split_dataset = chunked_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Chunked Train Dataset Size: {len(train_dataset)}")
print(f"Chunked Test Dataset Size: {len(test_dataset)}")

Chunked Train Dataset Size: 82
Chunked Test Dataset Size: 10


In [16]:
# Set up Data Collator for Dynamic Padding during training 
from transformers import DataCollatorForLanguageModeling
import os

# Supress TensorFlow warning/logging since we're using PyTorch anyway
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow logging

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

### Model Training

In [17]:
# Check available GPU
import torch

if torch.cuda.is_available():
    print(f"Available GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected.")


Available GPU: NVIDIA L40S


In [18]:
import os #disable unless we have parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [19]:
from transformers import AutoModelForCausalLM
import torch

# Load pre-trained model of choice - make sure it matches with AutoTokenizer model
print(f"Currently using this model: {model_name}")
#model = AutoModelForCausalLM.from_pretrained(model_name) 

#Compile model - optimizes moodel for faster execution
model = torch.compile(model)

# Move model to GPU 
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


Currently using this model: meta-llama/Llama-2-7b-hf


OptimizedModule(
  (_orig_mod): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(32000, 4096)
      (layers): ModuleList(
        (0-31): 32 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
            (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
            (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
          (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    

#### Model Training Strategy Notes
- Default is full fine-tuning!
    - All layers of the models are updated - takes lots of memory, time and compute --> better performance but risk of overfitting on smaller datasets.
- Alternative: **LoRA** - Low-Rank Adaptation
    - Freezes most of the model, only train a few lightweight adapter layers
    - Uses peft

- Recap/Overview of PyTorch TrainingArguments - configures/sets up how to train the model. "The recipe"
- Common Parameters ...
        - `output_dir="./model_output"` - Directory where the trained model & checkpoints are saved.
    - `num_train_epochs=3` - Number of times the model sees the full dataset.
    - `per_device_train_batch_size=8` - Batch size per GPU/CPU during training.
    - `per_device_eval_batch_size=8` - Batch size per GPU/CPU during evaluation.
    - `learning_rate=5e-5` - Step size for adjusting model weights.
    - `weight_decay=0.01` - Regularization to prevent overfitting.
    - `evaluation_strategy="epoch"` - When to evaluate (options: `"no"`, `"steps"`, `"epoch"`).
    - `save_strategy="epoch"` - When to save the model (options: `"no"`, `"steps"`, `"epoch"`).
    - `logging_steps=500` - How often to log training progress.
    - `save_total_limit=2` - Keeps only the latest 2 model checkpoints (deletes older ones).
    - `fp16` = 16-bit floating point precision aka **mixed precision training**
        - Reduces memory usage + speeds up training  by using half-precision 16-bit numbers instead of full precisiion 32-bit numbers. Helps with GPU training.
- Advanced Parameters (for Performance Tuning and Debugging)
    - `gradient_accumulation_steps=4` - Simulates larger batch sizes by accumulating gradients over multiple steps.
    - `warmup_steps=500` - Gradually increases learning rate at the beginning of training.
    - `logging_dir="./logs"` - Where to store training logs for visualization (e.g., TensorBoard).
    - `load_best_model_at_end=True` - Automatically loads the best model after training.
    - `metric_for_best_model="loss"` - Defines the metric used to determine the best model.
    - `disable_tqdm=True` - Disables the training progress bar (useful for notebooks or logging).
    - `adam_beta1=0.9, adam_beta2=0.999` - Custom AdamW optimizer settings.
    - `lr_scheduler_type="cosine"` - Defines learning rate decay (options: `"linear"`, `"cosine"`, `"constant"`).
    - `seed=42` - Sets a random seed for reproducibility.
    - `fp16_opt_level="O1"` - Further controls mixed precision behavior.

In [20]:
#Supress TF warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


In [21]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True


In [22]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=f"./marthabot_models/{model_name}_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # A100 can handle 16+ easily
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,  # No need to accumulate on A100
    num_train_epochs=10, #start with a small amount, Scott will help create script for large-scale training layer on
    weight_decay=0.01,
    save_total_limit=2,
    bf16=True, #fp16=True,  # Enable mixed precision for efficiency - Scott said start with this true
    logging_dir="./logs",
    logging_steps=50, #track progress --> monitor loss over time, detect over/underfitting early, debug errors
    report_to="none", #remove unwanted logging
    remove_unused_columns=False, #was getting error before in Training
)

- Recap/Overview of PyTorch Trainer class - handles and the runs the full training process... "The chef"
    - Model Training
    - Evaluation
    - Saving/loading models
    - Logging Training progress
- Common Parameters ...
    - `model=model` - The pre-trained model to fine-tune.
    - `args=training_args` - The training configuration (`TrainingArguments` object).
    - `train_dataset=train_dataset` - The dataset used for training.
    - `eval_dataset=test_dataset` - The dataset used for evaluation.
    - `data_collator=data_collator` - Handles dynamic padding for batches.
    - `tokenizer=tokenizer` - The tokenizer used for text preprocessing.
    - `compute_metrics=compute_metrics` - Custom function for calculating evaluation metrics.
- Advcaned Parameters (for Performance tuning and debugging)
    - `optimizers=(optimizer, scheduler)` - Custom optimizer (AdamW, SGD) and learning rate scheduler.
    - `callbacks=[callback]` - Custom callbacks for additional monitoring or stopping conditions.
    - `preprocess_logits_for_metrics=logits_processor` - Custom function to modify logits before computing metrics.
    - `disable_tqdm=True` - Disables progress bars (useful for logging environments).
    - `data_collator=default_data_collator` - Uses Hugging Faces default data collator.
    - `eval_accumulation_steps=10` - Stores evaluation results every 10 steps.
    - `max_length=1024` - Maximum sequence length for model inputs.
    - `gradient_checkpointing=True` - Reduces memory usage during training (useful for large models).


In [23]:
# Remove the text column
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

In [24]:
# Check dataset cols: 
print(train_dataset.column_names)

['input_ids', 'attention_mask', 'labels']


In [27]:
# Free up GPU memory
import torch
torch.cuda.empty_cache()

# Report memory in MB
allocated = torch.cuda.memory_allocated() / 1024**2
reserved = torch.cuda.memory_reserved() / 1024**2

print(f"Allocated: {allocated:.2f} MB")
print(f"Reserved: {reserved:.2f} MB")

Allocated: 44476.89 MB
Reserved: 44756.00 MB


In [28]:
# For other models:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset, 
    data_collator=data_collator,
    tokenizer = tokenizer,
)
trainer.train()

/scratch/3199537.1.cds-gpu/ipykernel_321811/2395594801.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacty of 44.40 GiB of which 66.12 MiB is free. Including non-PyTorch memory, this process has 44.33 GiB memory in use. Of the allocated memory 43.56 GiB is allocated by PyTorch, and 280.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
#Used CustomTrainer for gpt2 was getting some weird errors
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):  #  Accept extra kwargs
        # Remove unexpected arguments
        inputs.pop("num_items_in_batch", None)  #  Prevents the error
        
        outputs = model(**inputs)  # Forward pass
        loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

# Try using the CustomTrainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset, 
    data_collator=data_collator,
    tokenizer = tokenizer,
)

trainer.train()


In [ ]:
# Save the model
trainer.save_model(f"./marthabot_models/{model_name}_finetuned")
tokenizer.save_pretrained(f"./marthabot_models/{model_name}_finetuned")


### Text Generation + Inference + Evaluation

In [ ]:
#Supress TF warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
from transformers import pipeline
from transformers import AutoTokenizer

# Load fine-tuned model. MAKE SURE CORRECT MODEL NAME IS HERE
model_name_eval = "gpt2"
model_path = f"./marthabot_models/{model_name_eval}_finetuned/"
tokenizer = AutoTokenizer.from_pretrained(model_name_eval)

In [ ]:
# Generate Text
generator = pipeline("text-generation", model=model_path, tokenizer=tokenizer, pad_token_id = tokenizer.eos_token_id)

prompt = "I am a dancer. I believe that we learn by practice. Whether it means to learn to dance by"
for i, gen in enumerate(output[0]):  # =H output[0] is the list of generations for the first prompt
    print(f"=== Generation {i+1} ===")
    print(gen["generated_text"])
    print()

In [ ]:
# The original 
clean_text[0:400]

In [ ]:
import math 

eval_results = trainer.evaluate()
print(f"Validation Loss: {eval_results['eval_loss']}")

# Compute perplexity
perplexity = math.exp(eval_results["eval_loss"])
print(f"Perplexity: {perplexity}")

In [ ]:
# Make sure Cross-Entropy Loss is the evaluation metric. 
from transformers import GPT2LMHeadModel
import inspect

print(inspect.getsource(GPT2LMHeadModel.forward))
